In [ ]:
import pandas as pd
import duckdb
import sys
sys.path.insert(0, "..")  # per trovare il modulo vigipy locale

from vigipy import GPS
from vigipy.utils import Container
from src.contingency_table import build_contingency_table, qc_contingency_table

%run ./04_contingency_duck.ipynb


la cella seguente esegue MGPS quando ci sono abbastanza dati per clcolare i prior

In [ ]:
results = GPS.gps(
    container,
    relative_risk=1,
    min_events=3,           # soglia letteratura
    decision_metric="rank",
    ranking_statistic="log2",  # EBGM — metrica standard MGPS
    expected_method="mantel-haentzel",
)

print(f"Segnali rilevati: {results.num_signals}")
results.signals.head(20)

dato che ora stiamo lavorando con un subset, mettiamo manualmente i prior
Sono i prior empirici di DuMouchel (1999) — il paper originale che ha introdotto il GPS — stimati sull'intero database FAERS. Sono lo standard de facto quando si analizza un singolo drug e non si può stimare i prior dalla matrice di contingenza (che nel nostro caso ha una sola riga).


In [ ]:
results = GPS.gps(
    container,
    relative_risk=1,
    min_events=3,
    decision_metric="rank",
    ranking_statistic="log2",
    expected_method="mantel-haentzel",
    prior_param=[0.2041, 0.05816, 1.415, 1.838, 0.0969],  # prior FDA DuMouchel 1999
)

estrazione segnale positivo statisticamente significativo 

In [ ]:
# Colonne chiave per MGPS:
# log2   = EBGM (Empirical Bayes Geometric Mean) → soglia positività > 2
# LowerBound = EB05 (lower bound 90% CI)         → criterio letteratura: EB05 > 2
# p_value = probabilità posteriore H0

mgps_signals = results.all_signals[
    results.all_signals["LowerBound"] >= 2  # criterio standard
].copy()

print(f"Segnali con EB05 >= 2: {len(mgps_signals)}")
mgps_signals[["Adverse Event", "Count", "Expected Count", 
               "log2", "LowerBound", "p_value"]].head(20)